# X-ray Lab 2026 — NGC 300 ULX-1: spin-up of an ultraluminous X-ray pulsar

This notebook is **Part 1** of the lab notes (Chapter 2). Goal: measure the spin frequency $\nu$ of the neutron star and how fast it changes, $\dot\nu$, during a ~3.5-day NuSTAR observation.

**How to work**
- Run the cells in order (`Shift + Enter`).
- ✏️ **TODO** = you change or write one or two lines of code.
- ❓ = a question to think about; the answers help you with the report.
- Sections marked **EXTRA** at the end are optional.

At the end you will add your result to the **shared Google Sheet** of the class.

## 0. Software

We need `numpy`, `scipy`, `matplotlib` and `astropy`. If a package is missing, open a terminal, activate the lab environment and install it from conda-forge, e.g.

```
conda activate xray
conda install -c conda-forge numpy scipy matplotlib astropy
```

(`stingray` is only used in the EXTRA section: `conda install -c conda-forge stingray`.) After installing, restart the kernel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import scipy.optimize
from astropy.io import fits

plt.rcParams['figure.dpi'] = 100

## 1. Choose your data file

We have **four** event files of the same NuSTAR observation (December 2016):

| key | detector | barycentric correction | file |
|---|---|---|---|
| `A_bary`   | FPMA | yes | `nu30202035002A01_cl_bary.evt.gz` |
| `B_bary`   | FPMB | yes | `nu30202035002B01_cl_bary.evt.gz` |
| `A_nobary` | FPMA | no  | `nu30202035002A01_cl.evt.gz` |
| `B_nobary` | FPMB | no  | `nu30202035002B01_cl.evt.gz` |

NuSTAR has two identical telescopes with two detectors, FPMA and FPMB. In the barycentred files the photon arrival times have been converted to the barycentre of the Solar System (see the EXTRA section).

Each student analyses **one** file. When we put everybody's results together we can check whether the detector or the barycentric correction changes the measured $\dot\nu$.

✏️ **TODO:** set `my_choice` to the file you were assigned (or chose).

In [ ]:
files = {
    'A_bary':   'nu30202035002A01_cl_bary.evt.gz',
    'B_bary':   'nu30202035002B01_cl_bary.evt.gz',
    'A_nobary': 'nu30202035002A01_cl.evt.gz',
    'B_nobary': 'nu30202035002B01_cl.evt.gz',
}

my_choice = 'A_bary'     # TODO: one of 'A_bary', 'B_bary', 'A_nobary', 'B_nobary'

event_file = files[my_choice]
hdul = fits.open(event_file)
data = hdul[1].data          # the table of events (extension [1])
hdr = hdul[1].header       # information about the observation

print('File:', event_file)
for key in ['TELESCOP', 'OBJECT', 'DATE-OBS', 'TSTART', 'TSTOP', 'TIMEREF']:
    print(f'{key:9s} = {hdr.get(key)}')

❓ `TIMEREF = SOLARSYSTEM` means the times are barycentred, `LOCAL` means they are not. Does it agree with the file you chose?
Write down `DATE-OBS`: you need it in Part 2 to find simultaneous XMM-Newton observations.

## 2. What is in an event file?

A FITS event file is a table: **one row per detected photon**. Let us see which columns it has.

In [ ]:
print(data.columns.names)
print('number of events:', len(data))

✏️ **TODO:** print the content of the columns `PI` and `TIME` (e.g. `print(data['PI'])`), and their minimum and maximum (`.min()`, `.max()`).

❓ What do you think each column contains?
- `TIME`: arrival time of each photon, in seconds since a reference date of the mission (1 January 2010 for NuSTAR).
- `PI`: a channel number, proportional to the photon energy. For NuSTAR: $E = 0.04 \times \mathrm{PI} + 1.6$ keV.
- `X`, `Y`: position of the photon on the sky, in pixels.

In [ ]:
# TODO: print the PI and TIME columns and their min / max
print(data['PI ....

## 3. Image and source region

An image is a **2D histogram** of the photon positions `X`, `Y`: we count how many photons fall in each small square (bin).

In [ ]:
x, y = data['X'], data['Y']
cmap = mpl.colormaps['gray'].copy(); cmap.set_bad('black')   # empty bins shown black
H, xedges, yedges = np.histogram2d(x, y, bins=200, range=[[0, 1000], [0, 1000]])   # 5x5-pixel bins

plt.figure(figsize=(6, 5))
plt.imshow(H.T, origin='lower', extent=[0, 1000, 0, 1000], cmap=cmap, norm=mpl.colors.LogNorm())
plt.colorbar(label='counts per bin')
plt.xlim(300, 700); plt.ylim(300, 700)
plt.xlabel('X (pixel)'); plt.ylabel('Y (pixel)')
plt.show()

✏️ **TODO:** look at the image and estimate **by eye** the centre of the bright source (`xc`, `yc`) and a radius `rad` (in pixels) that contains most of its photons. Use `plt.xlim` / `plt.ylim` in the cell above to zoom in if you like. The two detectors (A, B) have slightly different coordinates, so do not copy the numbers of a classmate who uses the other detector!

*Side task (optional):* since the image is a histogram, the centre is roughly the bin with the most counts. Hint: `i, j = np.unravel_index(np.argmax(H), H.shape)` gives the indices of that bin, and its centre is `0.5*(xedges[i] + xedges[i+1])`, `0.5*(yedges[j] + yedges[j+1])`.
(The proper way would be to fit a 2D Gaussian (or the PSF) to the image; we do not do this in class.)

In [ ]:
xc, yc = 500.0, 500.0    # TODO: centre of the source, by eye (pixels)
rad = 30.0               # TODO: radius of the region (pixels)

# Side task (optional): brightest bin of the image

In [ ]:
def in_circle(xc, yc, rad, x, y):
    # True for events inside the circle, False outside
    return (x - xc)**2 + (y - yc)**2 <= rad**2

src = in_circle(xc, yc, rad, x, y)
print('events in file:', len(x), '   inside the circle:', src.sum())

plt.figure(figsize=(6, 5))
plt.imshow(H.T, origin='lower', extent=[0, 1000, 0, 1000], cmap=cmap, norm=mpl.colors.LogNorm())
plt.gca().add_patch(plt.Circle((xc, yc), rad, fill=False, color='red', lw=2))
plt.xlim(xc - 150, xc + 150); plt.ylim(yc - 150, yc + 150)
plt.xlabel('X (pixel)'); plt.ylabel('Y (pixel)')
plt.show()

## 4. Energy selection

We keep the photons inside the circle with energies between 3 and 30 keV. Below 3 keV NuSTAR is not sensitive, and above ~30 keV this source is faint compared with the background.

We also subtract the time of the first photon, so that times start from zero (in seconds).

In [ ]:
E = 0.04 * data['PI'] + 1.6                 # energy in keV
Emin, Emax = 3.0, ...TODO
sel = src & (E > Emin) & (E <= Emax)

t_ev = data['TIME'][sel]
t_ev = t_ev - t_ev.min()                  # time since the first photon (s)
T_obs = t_ev.max()

print('selected photons:', len(t_ev))
print(f'duration of the observation: {T_obs:.0f} s = {T_obs/86400:.2f} days')

## 5. Light curve

A light curve is a histogram of the arrival times: the number of photons in each time bin.

✏️ **TODO:** at the end of the cell, add **one line** that zooms into the first 20 ks (20000 s) of the observation. *Hint:* `plt.xlim(...)`.

❓ What do you see? How long does a gap last, and how often does it repeat? NuSTAR is in a low orbit around the Earth: what could cause the gaps?

In [ ]:
binsize = 100.0          # seconds
edges = np.arange(0, T_obs + binsize, binsize)
lc, _ = np.histogram(t_ev, bins=edges)

plt.figure(figsize=(12, 4))
plt.plot(0.5 * (edges[1:] + edges[:-1]), lc / binsize, drawstyle='steps-mid')
plt.xlabel('time since first photon (s)'); plt.ylabel('count rate (counts/s)')
# TODO: add one line here to zoom into the first 20 ks
plt.show()

## 6. Folding: the pulse profile

**Folding** (epoch folding): for a trial period $P$ we compute the **phase** of every photon, $\phi = (t/P) \bmod 1$ (a number between 0 and 1), and make a histogram of the phases with `nbin` bins. This is the average **pulse profile**.
- If $P$ is close to the true period, the photons pile up at the same phases: we see a clear pulse.
- If $P$ is wrong, the phases are random: the profile is flat.

The profile is plotted twice (phase 0–2) only to make it easier to see.

**Noise level.** If the profile were flat, each bin would contain on average $\bar N = N_{tot}/n_{bin}$ photons, with random (Poisson) fluctuations of about $\sqrt{\bar N}$. The **grey band** shows $\bar N \pm \sqrt{\bar N}$. A profile that stays inside the band is consistent with no pulsation (even for a flat profile, about 1/3 of the bins fall just outside the band); a real pulse goes well beyond it.

In [ ]:
def fold(times, period, nbin):
    phase = (times / period) % 1.0
    counts, _ = np.histogram(phase, bins=nbin, range=(0.0, 1.0))
    return counts

def plot_profile(counts, label=None):
    nbin = len(counts)
    phase = (np.arange(2 * nbin) + 0.5) / nbin
    c2 = np.concatenate([counts, counts])
    mean = counts.mean()                    # expected counts per bin if the profile were flat
    plt.axhspan(mean - np.sqrt(mean), mean + np.sqrt(mean), color='grey', alpha=0.15, zorder=0)
    plt.errorbar(phase, c2, yerr=np.sqrt(c2), drawstyle='steps-mid', label=label)
    plt.xlabel('phase'); plt.ylabel('counts per phase bin')

✏️ **TODO:**
1. Choose the number of phase bins `nbin` (try 8, 16, 32).
2. Our initial guess for the period is `P_guess = 31.6` s (from earlier observations).
3. Set `P_random` to different values and compare the two profiles: e.g. 25, 31, 31.5, 31.58, 31.62…

❓ How far from `P_guess` must `P_random` be for its profile to look **flat**? Compare the difference $|P_{random} - P_{guess}|$ with the number $P^2/T_{obs}$ printed below.
*Hint:* during the observation the source makes $N = T_{obs}/P$ rotations. If the trial period is wrong by $\Delta P$, the phase of the photons drifts by $N\,\Delta P/P = T_{obs}\,\Delta P/P^2$ cycles by the end. When this drift becomes ~1 cycle, the pulse is smeared out.

In [ ]:
nbin = 8                 # TODO: number of phase bins (8, 16, 32 ...)
P_guess = 31.6           # initial guess of the period (s)
P_random = ...          # TODO: try several values

plt.figure(figsize=(7, 4))
plot_profile(fold(t_ev, P_guess, nbin), label=f'P = {P_guess} s')
plot_profile(fold(t_ev, P_random, nbin), label=f'P = {P_random} s')
plt.legend()
plt.show()

print(f'P^2 / T_obs = {P_guess**2 / T_obs:.4f} s')

## 7. Period search over a range of frequencies

Instead of trying periods one by one, we scan many trial frequencies $f = 1/P$ and give each one a **score**: how far the folded profile is from flat. As a score we use the **standard deviation (rms)** of the counts in the phase bins:
- flat profile → only random (Poisson) fluctuations → small rms;
- pulsed profile → large differences between bins → large rms.

The best frequency is the one with the **maximum** score. (This is equivalent to the usual $\chi^2$ test against a flat profile.)

**Frequency step.** Two frequencies that differ by less than $1/T_{obs}$ give almost the same profile (the phase drift is less than one cycle, see Section 6). So $1/T_{obs}$ is the natural resolution. We use a step $df = 1/(\text{oversample} \times T_{obs})$, i.e. a few points per resolution element.

In [ ]:
def period_search(times, freqs, nbin):
    # rms του αναδιπλωμένου προφίλ για κάθε δοκιμαστική συχνότητα
    score = np.zeros(len(freqs))
    for k in range(len(freqs)):
        f = freqs[k]
        phase = (times * f) % 1.0
        counts, _edges = np.histogram(phase, bins=nbin, range=(0.0, 1.0))
        score[k] = np.std(counts)
    return score

f_guess = 1 / P_guess
oversample = 5           # TODO: try 1, 5, 0.2 and see what happens
df = 1 / (oversample * T_obs)
freqs = np.arange(f_guess - 1e-3, f_guess + 1e-3, df)

score_all = period_search(t_ev, freqs, nbin)

plt.figure(figsize=(8, 4))
plt.plot(freqs, score_all)
plt.axvline(f_guess, ls=':', color='grey', label='f_guess')
plt.xlabel('frequency (Hz)'); plt.ylabel('rms of folded profile (counts)')
plt.legend(); plt.show()
print('best frequency (all data):', freqs[np.argmax(score_all)], 'Hz')

❓ The periodogram of the whole observation has **several** peaks and is not a single sharp line. Two reasons:
1. the frequency of the pulsar **changes** during the observation (this is what we want to measure!);
2. the data have regular **gaps** (Section 5).

So we cannot use all the data at once: we will split them into shorter segments.

## 8. Selecting part of the data: masks and aliasing

To select the photons of a time interval we use a **mask**: an array of `True`/`False`, one for each photon. Below we keep only the first 30 ks and search for the period there.

**Side peaks (aliasing).** The periodogram of a single segment shows a main peak and smaller peaks on both sides. They come from the **orbital gaps**: the data are "switched on and off" every NuSTAR orbit ($P_{orb} \approx 96$ min $\approx 5800$ s). Because of this regular window, the signal also appears at frequencies $f \pm k/P_{orb}$, with $1/P_{orb} \approx 1.7\times10^{-4}$ Hz. The dashed lines mark $f_{peak} \pm 1/P_{orb}$. We will discuss this on the board.

In [ ]:
P_orb = 5800           # NuSTAR orbital period (s)

mask1 = (t_ev >= 0) & (t_ev < 30e3)       # first 30 ks
t1 = t_ev[mask1]
freqs1 = np.arange(f_guess - 1e-3, f_guess + 1e-3, 1 / (oversample * (t1.max() - t1.min())))
score1 = period_search(t1, freqs1, nbin)
f1 = freqs1[np.argmax(score1)]

plt.figure(figsize=(8, 4))
plt.plot(freqs1, score1, label='0-30 ks')
for k in (-1, 1):
    plt.axvline(f1 + k / P_orb, ls='--', color='grey',label='alias')
plt.axvline(f1, ls=':', color='magenta',label='best')
plt.xlabel('frequency (Hz)'); plt.ylabel('rms of folded profile (counts)')
plt.legend(); plt.show()
print(f'0-30 ks: f = {f1:.7f} Hz,  P = {1/f1:.4f} s')

✏️ **TODO:** in the empty cell below, do the same for the interval **270–300 ks**: make `mask2`, `t2`, `freqs2`, `score2`, `f2`, and plot `score1` and `score2` **on the same figure** (copy and change the cell above).

❓ Does the main peak move to higher or lower frequency? By how much? From $\Delta f$ and the time between the two intervals, make a first rough estimate of $\dot\nu \approx \Delta f / \Delta t$. Is the pulsar spinning up or down?

In [ ]:
# TODO: interval 270-300 ks, plot together with 0-30 ks

## 9. Splitting the observation into N segments

The function below splits the data into `n_seg` segments of equal duration, runs the period search in each one and keeps the frequency of the maximum.

✏️ **TODO:**
1. Choose `n_seg` (try e.g. 4, 8, 12) and the number of phase bins `nbin_seg`.
2. Look at the periodogram of **every** segment (grid of small plots). The red line marks the frequency that was chosen. If in some segment the code picked a **side peak (alias)** instead of the main peak, write the number of that segment in the list `exclude` (e.g. `exclude = [3]`) and run the cell again.

❓ What is the trade-off when you choose `n_seg`? More segments give more points for the fit, but each segment is shorter: fewer photons and a broader peak (resolution $1/T_{seg}$).

**Write down** the `n_seg` and `nbin_seg` you finally used: they go into the Google Sheet.

In [ ]:
def search_segments(t, n_seg, nbin, f_guess, oversample=5, half_width=1e-3):
    edges = np.linspace(t.min(), t.max(), n_seg + 1)
    T_mid, F_best, pgrams = [], [], []
    for k in range(n_seg):
        m = (t >= edges[k]) & (t < edges[k + 1])
        ts = t[m]
        fr = np.arange(f_guess - half_width, f_guess + half_width, 1 / (oversample * (ts.max() - ts.min())))
        sc = period_search(ts, fr, nbin)
        T_mid.append(ts.mean())
        F_best.append(fr[np.argmax(sc)])
        pgrams.append((fr, sc))
    return np.array(T_mid), np.array(F_best), pgrams

In [ ]:
n_seg = 1                # TODO: number of segments
nbin_seg = 10            # TODO: number of phase bins

T_mid, F_best, pgrams = search_segments(t_ev, n_seg, nbin_seg, f_guess)

ncol = 4
nrow = int(np.ceil(n_seg / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(14, 2.6 * nrow), squeeze=False)
for k, (fr, sc) in enumerate(pgrams):
    ax = axes.flat[k]
    ax.plot(fr * 1e3, sc)
    ax.axvline(F_best[k] * 1e3, color='red')
    ax.set_title(f'segment {k}  (t = {T_mid[k]/1e3:.0f} ks)', fontsize=9)
for ax in axes.flat[n_seg:]:
    ax.axis('off')
fig.supxlabel('frequency (mHz)'); fig.supylabel('rms')
plt.tight_layout(); plt.show()

T_data, F_data = T_mid, F_best

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(T_data / 1e3, F_data * 1e3, 'o')
plt.xlabel('time since first photon (ks)'); plt.ylabel('frequency (mHz)')
plt.show()

## 10. Linear fit: $\nu(t) = \dot\nu\,(t - t_{ref}) + \nu_0$

We fit a straight line $y = m\,(t - t_{ref}) + b$ to the points: the slope $m$ is $\dot\nu$ and $b$ is the frequency $\nu_0$ at the reference time $t_{ref}$. We choose $t_{ref}$ at the middle of the observation.

<details><summary><b>How the fit and its errors work (optional reading)</b></summary>

`scipy.optimize.curve_fit` finds the parameters that minimise the sum of squared residuals
$$S(m,b) = \sum_i \left[y_i - \big(m\,(t_i - t_{ref}) + b\big)\right]^2 .$$
It returns the best-fit values and the **covariance matrix** `pcov`:
$$\mathrm{pcov} = \begin{pmatrix}\sigma_m^2 & \mathrm{cov}(m,b)\\ \mathrm{cov}(m,b) & \sigma_b^2\end{pmatrix}.$$
The diagonal elements are the **variances** of the parameters, so the errors are $\sigma_m = \sqrt{\mathrm{pcov}[0,0]}$ and $\sigma_b = \sqrt{\mathrm{pcov}[1,1]}$. The off-diagonal element tells us how much the two parameters are correlated; choosing $t_{ref}$ in the middle of the data makes it close to zero.

Because we do not give errors for the individual points, `curve_fit` estimates the scatter of the points from the residuals, $s^2 = S_{min}/(N-2)$ ($N$ points, 2 parameters), and scales the errors with it. So the errors reflect how well the points follow a straight line.
</details>

In [ ]:
t_ref = 0.5 * T_obs

def lin(t, m, b):
    return m * (t - t_ref) + b

fit_params, pcov = scipy.optimize.curve_fit(lin, T_data, F_data, p0=[0.0, f_guess])
m, b = fit_params
m_err, b_err = np.sqrt(np.diag(pcov))

F_fit = T_data*m +b #predicted Freq by best fit model

print(f'nudot = m = {m:.3e} +/- {m_err:.1e} Hz/s')
print(f'nu0   = b = {b:.7f} +/- {b_err:.1e} Hz   (at t_ref = {t_ref:.0f} s)')

## 11. Final plot (for the report)

✏️ **TODO:**
1. On the upper panel, plot the **best-fit line** using `m` and `b` (careful with the units of the axes: time in ks, frequency in mHz). The array `tt` contains times for drawing the line.
2. Compute the **residuals** (data − model) for every point: replace the line `resid = np.zeros_like(F_fit)`.

This figure must be included in your report.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True, gridspec_kw=dict(height_ratios=[3, 1]))

ax1.plot(T_data / 1e3, F_data * 1e3, 'o', label='segments')
tt = np.linspace(0, T_obs, 200)
# TODO: plot the best-fit line on ax1 (time in ks, frequency in mHz)
ax1.set_ylabel('frequency (mHz)')
ax1.legend()

# TODO: residuals = data - model, for each point
resid = np.zeros_like(F_fit) # <-TODO define residuals as (data − model)

ax2.plot(T_data / 1e3, resid * 1e6, 'o')
ax2.axhline(0, color='grey', lw=1)
ax2.set_xlabel('time since first photon (ks)')
ax2.set_ylabel(r'residual ($\mu$Hz)')
plt.show()

## 12. Results for the shared Google Sheet

The period derivative follows from $P = 1/\nu$: $\dot P = -\dot\nu/\nu^2$, with error $\sigma_{\dot P} = \sigma_{\dot\nu}/\nu^2$.

✏️ **TODO:** fill in your name and student number and run the cell. The last line is **tab-separated**: copy it and paste it into the first empty row of the Google Sheet, and the values will go into separate columns. Check that the columns match the header of the sheet.

In [ ]:
student_name = ''        # TODO: your name
student_id = ''          # TODO: your student number

Pdot = -m / b**2
Pdot_err = m_err / b**2
print(f'P     = {1/b:.5f} s')
print(f'nudot = {m:.3e} +/- {m_err:.1e} Hz/s')
print(f'Pdot  = {Pdot:.3e} +/- {Pdot_err:.1e} s/s')
print()
print('name | student no. | file | n_seg | nbin | nudot | nudot_err | Pdot | Pdot_err')
print('\t'.join([student_name, student_id, my_choice, str(n_seg), str(nbin_seg),
                 f'{m:.4e}', f'{m_err:.2e}', f'{Pdot:.4e}', f'{Pdot_err:.2e}']))

---
# EXTRA A: magnetic field and uncertainties (Part 3 of the notes)

From the lab notes, eq. [7] (cgs units):
$$B_s = 2^{1/4}\,(2\pi)^{7/2}\; G^{-3/2}\, M^{-3/2}\, R^{-3}\, I^{7/2}\, \xi^{-7/4}\, \dot\nu^{7/2}\, \dot m^{-3},
\qquad \dot m = \frac{L R}{G M},\qquad L = 4\pi d^2 F.$$

✏️ **TODO:** enter the distance you chose in Part 2 and its error. The flux is the broadband value of Carpano et al. (2018), with a 10% error.

In [ ]:
G = 6.674e-8; Msun = 1.989e33; yr = 3.156e7; Mpc = 3.086e24      # cgs

def B_surface(nudot, mdot, M, R, I, xi):
    # eq. [7] of the lab notes (cgs)
    return (2**0.25 * (2*np.pi)**3.5 * G**-1.5 * M**-1.5 * R**-3.0
            * I**3.5 * xi**-1.75 * nudot**3.5 * mdot**-3.0)

d_Mpc, d_err = 1.88, 0.05             # TODO: your distance and its error (Mpc)
F, F_err = 1.1e-11, 0.11e-11          # erg/s/cm^2 (Carpano et al. 2018, 10%)
M, R, xi = 1.4 * Msun, 1.0e6, 0.5
I = 0.4 * M * R**2

L = 4 * np.pi * (d_Mpc * Mpc)**2 * F
mdot = L * R / (G * M)
B = B_surface(m, mdot, M, R, I, xi)
print(f'L = {L:.2e} erg/s,  mdot = {mdot:.2e} g/s = {mdot*yr/Msun:.2e} Msun/yr')
print(f'B = {B:.2e} G')

### Error propagation

For a product of powers, $B \propto \dot\nu^{7/2}\,\dot m^{-3}$, the **relative** errors add in quadrature, each multiplied by its exponent:
$$\left(\frac{\sigma_B}{B}\right)^2 = \left(\frac{7}{2}\,\frac{\sigma_{\dot\nu}}{\dot\nu}\right)^2 + \left(3\,\frac{\sigma_{\dot m}}{\dot m}\right)^2,
\qquad
\left(\frac{\sigma_{\dot m}}{\dot m}\right)^2 = \left(2\,\frac{\sigma_d}{d}\right)^2 + \left(\frac{\sigma_F}{F}\right)^2 .$$

✏️ **TODO:** compute `rel_mdot` and `rel_B` with these formulas.

In [ ]:
rel_mdot = 0.0           # TODO: relative error of mdot
rel_B = 0.0              # TODO: relative error of B
print(f'B = {B:.2e} +/- {rel_B*B:.1e} G   ({100*rel_B:.0f}%)')

---
# EXTRA B: does the barycentric correction matter here?

The observer moves: the Earth orbits the Sun at ~30 km/s. Because of the Doppler effect we measure $\nu_{obs} \simeq \nu\,(1 + v_\parallel/c)$, where $v_\parallel$ is the velocity of the observer towards the source. If $v_\parallel$ changes during the observation, we also measure a false frequency derivative,
$$\dot\nu_{app} \simeq \frac{\nu}{c}\,\frac{dv_\parallel}{dt}.$$
The barycentric correction removes these effects. Below we compute, with `astropy`, the actual velocity of the Earth towards NGC 300 ULX-1 during the observation.

❓ Is the effect important compared with your $\dot\nu$ and its error? Compare with the results in the Google Sheet from files with and without the correction. For what kind of analysis would the correction be essential? (Hint: the light-travel time changes by tens of seconds, about one pulse period.)

In [ ]:
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord, get_body_barycentric_posvel

c_kms = 299792.458
source = SkyCoord(13.7703 * u.deg, -37.6955 * u.deg)   # NGC 300 ULX-1 / SN 2010da
n_hat = source.cartesian.xyz.value

t_grid = Time(hdr['DATE-OBS'], scale='utc') + np.linspace(0, T_obs, 200) * u.s
pos, vel = get_body_barycentric_posvel('earth', t_grid)
v_los = vel.xyz.to(u.km / u.s).value.T @ n_hat           # km/s
r_los = pos.xyz.to(u.km).value.T @ n_hat                 # km

nudot_app = b * (v_los[-1] - v_los[0]) / T_obs / c_kms
print(f'Earth velocity towards the source: {v_los[0]:.2f} -> {v_los[-1]:.2f} km/s')
print(f'apparent nudot: {nudot_app:.1e} Hz/s   (your nudot: {m:.2e} +/- {m_err:.1e} Hz/s)')
print(f'worst case (a_Earth along the line of sight): {b * 29.8**2 / 1.496e8 / c_kms:.1e} Hz/s')
print(f'change of light-travel time during the observation: {(r_los.max() - r_los.min()) / c_kms:.1f} s')

---
# EXTRA C: comparison with Stingray

[Stingray](https://docs.stingray.science) is a Python package for X-ray timing analysis. It contains ready-made versions of what we wrote ourselves: `epoch_folding_search` (the $\chi^2$ version of our rms score) and `z_n_search` (the $Z^2_n$ statistic, Buccheri et al. 1983, which is sensitive to profiles made of a few sinusoidal harmonics). Below we compare them with our `period_search` on the first 30 ks.

In [ ]:
try:
    from stingray.pulse.search import epoch_folding_search, z_n_search
except ImportError:
    print('stingray is not installed: conda install -c conda-forge stingray')
else:
    _, ef = epoch_folding_search(t1, freqs1, nbin=nbin)
    _, zn = z_n_search(t1, freqs1, nbin=40, nharm=2)
    plt.figure(figsize=(8, 4))
    plt.plot(freqs1, score1 / score1.max(), label='our rms score')
    plt.plot(freqs1, ef / ef.max(), ls='--', label='stingray epoch folding')
    plt.plot(freqs1, zn / zn.max(), ls=':', label=r'stingray $Z^2_2$')
    plt.xlabel('frequency (Hz)'); plt.ylabel('normalised statistic'); plt.legend()
    plt.show()